In [116]:
# package imports:
import pandas as pd

# Core Data Workflow
This consistutes the core data workflow for beginning our data processing workflow through WRDS. We primarily query WRDS through the online portal (different access points) and run data operations/file conversions through this notebook and the */processes* folder, which contains .py files designed to consolidate workflows performed here.

We begin by using the WRDS query to gather info on the following SIC codes:
* 3674 — Semiconductors
* 3672 — Printed Circuit Boards
* 3679 — Electronic Components
* 2836 — Biological Products / Pharmaceuticals
* 2834 — Pharmaceutical Preparations
* 8731 — Commercial Physical & Biological Research
* 3571 — Electronic Computers
* 3572 — Computer Storage Devices
* 7372 — Prepackaged Software
* 7371 — Computer Programming Services

These are stored in our *raw_compustat_sic_pull* file under the *csv_data* folder in Data Wrangling. We pull data from Jan 2015 to Dec 2025.

In [117]:
# read in the raw compustat pull and see our basic dataframe shape. 
raw_compustat_pull = pd.read_csv("csv_data/raw_compustat_sic_pull.csv")
print(raw_compustat_pull.shape)

(13394, 21)


We now run a consolidation scrypt that isolates the companies with a market cap larger than $1 billion.

In [118]:
# market_cap_consolidation.py isolates companies with market cap larger than $1B and stores in a new csv
%run processes/market_cap_consolidation.py

In [119]:
# gather our new csv file and print the shape
compustat_large_cap = pd.read_csv("csv_data/compustat_large_cap.csv")
print(compustat_large_cap.shape)

(1463, 23)


In [120]:
# check that our smallest 
print(compustat_large_cap['mkt_cap'].min())

1006.04517


We have confirmed that our smallest company now has a market cap larger than 1 billion.

In [122]:
compustat_large_cap.groupby('industry')['tic'].nunique().reset_index().rename(columns={'tic': 'unique_tickers'})

,industry,unique_tickers
0,Biotech,85
1,Semiconductors/Hardware,75
2,Software,59


This brings us down to 219 companies across our three industry buckets.

In [ ]:
# save this version down as a csv
compustat_large_cap.to_csv("csv_data/compustat_large_cap_v2.csv", index=False)

In [124]:
%run processes/ticker_txt_conversion.py

Now we are going to run a scrypt that takes our tickers and finds the BoardEx ids. We do this through a basic BoardEx query in WRDS that inputs our tickers and only withdraw the boardex company id information. Afterwards, we check how many companies are left.

In [99]:
# boardex_id is our initial data pull and we see how many companies ultimately appear in boardex.
boardex_id = pd.read_csv("csv_data/boardex_id_data.csv")
len(boardex_id)

190

190 of our 1B+ market cap companies exist in boardex across the 3 industries. Now we want to store these company ids by themselves in a txt file so that we can withdraw more valuable information about key personnel for each of the companies.

In [ ]:
# run scrypt that eliminates cannabis companies and converts boardex ids down to a txt file for entry into WRDS.
%run processes/boardid_txt_conversion.py

### 3. [WRDS] Insert tickers into BoardEx and extract key personnel from 2015-2025

[a] Go to WRDS > BoardEx > North America > Organization Summary > Composition of Officers, Directors, & Senior Managers

[b] Date range of 2015 to end of 2025. Apply codes by ticker and input txt file copied from our downloaded csv and pasted into textedit for txt file conversion.

[c] Choose query variables (all) about officers and run query for csv file. Download csv file into local folder for input in step **4**.

In [127]:
# we pull our csv of the total boardex makeup
personnel = pd.read_csv("csv_data/total_board.csv")

In [128]:
# how many individuals do we actually have?
len(personnel)

21210

To consolidate to only CEOs, we need to isolate by rolename containing ceo, and seniority being an executive director (to avoid regional CEOs of large corporates)

In [129]:
ceos = personnel[personnel['rolename'].str.contains('ceo', case=False, na=False)]
ceos = ceos.sort_values('dateendrole', ascending=False) # want the most recent entries for each CEO first
ceos.head(3)

,companyid,datestartrole,directorid,directorname,companyname,rolename,dateendrole,datestartroleflag,dateendroleflag,seniority
12084,29816,2024-01-01,1864086,Sassine Ghazi,SYNOPSYS INC,President/CEO,9000-01-01,10,40,Executive Director
5180,16949,2024-06-05,1931869,Doctor Mark Gitin,IPG PHOTONICS CORP,CEO,9000-01-01,10,40,Executive Director
10850,26104,2023-12-11,340014,John Giamatteo,BLACKBERRY LTD (Research In Motion Ltd prior t...,CEO/Division President,9000-01-01,10,40,Executive Director


As we can see above, John Giamatteo is a regional/division president so we isolate by seniority set to executive director to only have top CEOs in our dataset.

In [130]:
ceos = ceos[ceos['seniority'] == 'Executive Director']

In [131]:
# how many CEOs do we have for our desired companies from 2015-2025?
len(ceos)

468

In [132]:
# save down our csv file of CEOs
ceos.to_csv("csv_data/ceos.csv", index=False)

In [133]:
# grab our consolidated director ids for additional boardex education queries
%run processes/directorid_txt_conversion.py

Now we proceed to extracting raw education data. For each of our 468 individual profiles from BoardEx, we are going to extract all education parameters available in BoardEx through WRDS.

In [134]:
raw_education = pd.read_csv('csv_data/boardex_raw_educ_pull.csv')

# let's visualize the education data that we are working with
raw_education.columns

Index(['directorid', 'directorname', 'companyname', 'awarddate',
       'qualification', 'fulltextdescription', 'primarykeyid', 'companyid',
       'awarddateflag'],
      dtype='object')

To correctly source, we are going to utilize some LLM magic. We print every possible manifestation of a school *qualification* (which is our degree type raw input) and every possible manifestation of *companyname* (which is our institution type). We then ask an LLM (claude opus 4.7) to correctly sort all of our degree strings into different categories for undergrad, master's, MD, MBA, and PHD. We also ask it to gather all instances of the top 20 US schools based on US News rankings. 

In [109]:
print(raw_education['companyname'].unique().tolist())

['Canadian Institute of Chartered Accountants (CICA)', "Queen's University", 'American Institute of Certified Public Accountants (AICPA)', 'Tuck School of Business Dartmouth', 'University of Southern California (USC)', 'Union College', 'London Business School', 'University of Edinburgh', 'Universiteit van Amsterdam (University of Amsterdam)', 'Pennsylvania State University (Penn State University)', 'Harvard Law School Harvard University', 'University of Texas at Austin (UT Austin)', "École Special des Travaux Publics du Batiment et de l'Industrie (ESTP)", 'Columbia Business School Columbia University', 'Franklin College', 'Marian University', 'University of Indianapolis', 'National University of Ireland', 'Indiana University', 'Harvard University', 'Xavier University Ohio', 'Purdue University', 'University of Chicago', 'Carey Law School University of Pennsylvania (Formerly known as University of Pennsylvania Law School)', 'Wharton School (The) University of Pennsylvania', 'Carnegie Mel

In [135]:
print(raw_education['qualification'].unique().tolist())

['Chartered Accountant', 'Degree', 'Certified Public Accountant', 'MBA (Distinction)', 'BSc (Hons)', 'BS', 'Fellow', 'MA', 'MBA', 'Diploma', 'BA', 'JD', 'Attended', 'MS', 'Doctorate (Hons)', 'PhD', 'BS (cum laude)', 'MA (Hons)', 'MD', 'BA (Hons)', 'Stanford Executive Program', 'BSEE', 'MSEE', 'JD (Hons)', 'BBA', 'JD (summa Cum Laude)', 'BS (Hons)', 'Graduated', 'Doctor of Humane Letters', 'Postdoctoral Fellow', 'AB', 'MSc', 'BSc', "Bachelor's Degree", 'Studied', 'BSME', 'MSME', 'AB (magna cum laude)', 'BSEE (magna cum laude)', 'BA (summa cum laude)', 'Certified', 'AB (Hons)', 'MEng', 'Post Graduate Diploma', 'BCom', 'Executive Program', 'Doctor of Science', 'Executive Development Program', 'Training Program', 'BSE', 'BSc (cum laude)', 'Certified Accountant', 'BS (Distinction)', 'Doctorate', 'Professional Development Program (PDP)', 'ME', 'BS (magna Cum Laude)', 'AA', 'Bachelor of Applied Science', 'BS (summa Cum Laude)', 'LLM', 'Completed', 'Postgraduate Studies', 'Certified Project Ma

After entering each of the above into our string, we build our respective lists. Then we perform a function that classifies each CEO in one row (instead of one row for every degree). Each row has a degree type and whether the CEO received that particular degree from a top 20 school, if applicable.

In [136]:
top20 = [
    # Harvard
    'Harvard University',
    'Harvard Business School Harvard University',
    'Harvard Medical School Harvard University',
    'Harvard Law School Harvard University',
    'Harvard TH Chan School of Public Health Harvard University',
    # Stanford
    'Stanford University',
    'Stanford Graduate School of Business',
    'Stanford University School of Medicine',
    # MIT
    'Massachusetts Institute of Technology (MIT)',
    'Massachusetts Institute of Technology (MIT) Sloan School of Management',
    # Princeton
    'Princeton University',
    # Yale
    'Yale University',
    'Yale School of Medicine (Yale-New Haven Medical Center)',
    # UChicago
    'University of Chicago',
    'University of Chicago Booth School of Business (University of Chicago Graduate School of Business prior to 2008)',
    'University of Chicago Law School',
    # Duke
    'Duke University',
    # Johns Hopkins
    'Johns Hopkins University',
    # Northwestern
    'Northwestern University',
    'Kellogg School of Management Northwestern University',
    'Northwestern University Pritzker School of Law',
    # UPenn
    'University of Pennsylvania',
    'Wharton School (The) University of Pennsylvania',
    'Carey Law School University of Pennsylvania (Formerly known as University of Pennsylvania Law School)',
    # Caltech
    'California Institute of Technology (CALTECH)',
    # Cornell
    'Cornell University \xa0New York',
    'Samuel Curtis Johnson Graduate School of Management Cornell University',
    # Brown
    'Brown University',
    # Dartmouth
    'Dartmouth College',
    'Tuck School of Business Dartmouth',
    # Columbia
    'Columbia University',
    'Columbia Business School Columbia University',
    # UC Berkeley
    'University of California Berkeley',
    'Haas School of Business University of California Berkeley',
    # Rice
    'Rice University',
    # UCLA
    'University of California Los Angeles (UCLA)',
    'UCLA Anderson School of Management',
    'UCLA School of Law',
    # Vanderbilt
    'Vanderbilt University',
    # Carnegie Mellon
    'Carnegie Mellon University',
]

ug_quals = {
    'Degree', 'BSc (Hons)', 'BS', 'BA', 'BS (cum laude)', 'BA (Hons)',
    'BSEE', 'BBA', 'BS (Hons)', 'AB', 'BSc', "Bachelor's Degree", 'BSME',
    'AB (magna cum laude)', 'BSEE (magna cum laude)', 'BA (summa cum laude)',
    'AB (Hons)', 'BCom', 'BSE', 'BSc (cum laude)', 'BS (Distinction)',
    'BS (magna Cum Laude)', 'BS (summa Cum Laude)', 'BTech',
    'Bachelor of Applied Science', 'BE (Hons)', 'Bachelor of Technology',
    "Bachelor's Degree (magna cum laude)", 'BEng (Hons)', 'LLB', 'BEng',
    'BPhil', 'BBA (Hons)', 'BBA (magna cum laude)', 'BE',
    "Bachelor's Degree (Hons)", 'BBA (summa cum laude)', 'AA'
}

mba_quals = {
    'MBA', 'MBA (Distinction)', 'MBA (Hons)', 'MBA (summa cum laude)',
    'International Executive MBA', 'BBA'
}

masters_quals = {
    'MA', 'MS', 'MA (Hons)', 'MSEE', 'MSc', 'MEng', 'MSME',
    'Masters Degree', 'Master of Management (MM)', 'LLM',
    'Master of Science (MOS)', 'MSc (magna cum laude)',
    'MSc (summa cum laude)', 'MBChB'
}

phd_quals = {
    'PhD', 'Doctorate', 'Doctorate (Hons)', 'Doctor of Humane Letters',
    'Doctor of Science', 'DSc', 'Doctor of Law (Hons)',
    'PhD (summa cum laude)', 'Doctor of Medicine (DM)'
}

md_quals = {
    'MD', 'MD (Hons)', 'Doctor of Veterinary Medicine (DVM)',
    'Bachelor of Medical Sciences (BMS)'
}

records = {}

for index, row in raw_education.iterrows():
    director_id = row['directorid']
    qual = row['qualification']
    company = row['companyname']
    institution = row['companyname']
    value = f"{qual} {company}"
    is_top20 = 1 if institution in top20 else 0

    if director_id not in records:
        records[director_id] = {
            'directorid': director_id,
            'UG': 0, 'top20_ug': 0,
            'MBA': 0, 'top20_mba': 0,
            'PhD': 0, 'top20_phd': 0,
            'MD': 0, 'top20_md': 0,
            "Master's": 0, 'top20_masters': 0
        }

    if qual in ug_quals:
        records[director_id]['UG'] = 1
        records[director_id]['top20_ug'] = max(records[director_id]['top20_ug'], is_top20)
    elif qual in mba_quals:
        records[director_id]['MBA'] = 1
        records[director_id]['top20_mba'] = max(records[director_id]['top20_mba'], is_top20)
    elif qual in phd_quals:
        records[director_id]['PhD'] = 1
        records[director_id]['top20_phd'] = max(records[director_id]['top20_phd'], is_top20)
    elif qual in md_quals:
        records[director_id]['MD'] = 1
        records[director_id]['top20_md'] = max(records[director_id]['top20_md'], is_top20)
    elif qual in masters_quals:
        records[director_id]["Master's"] = 1
        records[director_id]['top20_masters'] = max(records[director_id]['top20_masters'], is_top20)

education = pd.DataFrame.from_dict(records, orient='index').reset_index(drop=True)

In [137]:
# visualize what our operations above have actually done
education.describe()

,directorid,UG,top20_ug,MBA,top20_mba,PhD,top20_phd,MD,top20_md,Master's,top20_masters
count,3.180000e+02,318.000000,318.000000,318.000000,318.000000,318.000000,318.000000,318.000000,318.000000,318.000000,318.000000
mean,8.412866e+05,0.867925,0.179245,0.377358,0.169811,0.166667,0.059748,0.047170,0.012579,0.308176,0.110063
std,7.123928e+05,0.339106,0.384162,0.485490,0.376059,0.373265,0.237394,0.212336,0.111622,0.462467,0.313461
min,1.026000e+03,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,2.226352e+05,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,5.510355e+05,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
75%,1.268315e+06,1.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000
max,3.082984e+06,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000


We have now successfully mirrored King's data collection process. For example, we know that 86.7% of sample CEOs have some form of LLM-classified UG degree, with 17.9% of those particular degrees coming from top 20 undergraduate institutions. The same carries out for the rest of our specified degree types.

In [ ]:
# save down our updated csv file with education data present
education.sort_values(by='directorid', inplace=True)
education.to_csv("csv_data/EDUCATION_ORGANIZED.csv", index=False)

You may proceed to the *data_merge_1* file in this folder to see the next steps of our analysis.